In [ ]:
import pandas as pd
df = pd.read_csv('city_day.csv')
df.shape
df.info()
df.describe()
df['City'].nunique()
df['AQI_Bucket'].unique()

In [ ]:
import requests
import time

# Load your dataset
df = pd.read_csv('city_day.csv')
df['Date'] = pd.to_datetime(df['Date'])

# COMPLETE Dictionary of Coordinates for all cities in your dataset
city_coords = {
    'Ahmedabad': {'lat': 23.0225, 'lon': 72.5714},
    'Aizawl': {'lat': 23.7271, 'lon': 92.7176},
    'Amaravati': {'lat': 16.5131, 'lon': 80.5165},
    'Amritsar': {'lat': 31.6340, 'lon': 74.8723},
    'Bengaluru': {'lat': 12.9716, 'lon': 77.5946},
    'Bhopal': {'lat': 23.2599, 'lon': 77.4126},
    'Brajrajnagar': {'lat': 21.8216, 'lon': 83.9216},
    'Chandigarh': {'lat': 30.7333, 'lon': 76.7794},
    'Chennai': {'lat': 13.0827, 'lon': 80.2707},
    'Coimbatore': {'lat': 11.0168, 'lon': 76.9558},
    'Delhi': {'lat': 28.6139, 'lon': 77.2090},
    'Ernakulam': {'lat': 9.9816, 'lon': 76.2999},
    'Gurugram': {'lat': 28.4601, 'lon': 77.0199},
    'Guwahati': {'lat': 26.1445, 'lon': 91.7362},
    'Hyderabad': {'lat': 17.3850, 'lon': 78.4867},
    'Jaipur': {'lat': 26.9124, 'lon': 75.7873},
    'Jorapokhar': {'lat': 23.7004, 'lon': 86.4125},
    'Kochi': {'lat': 9.9312, 'lon': 76.2673},
    'Kolkata': {'lat': 22.5726, 'lon': 88.3639},
    'Lucknow': {'lat': 26.8467, 'lon': 80.9462},
    'Mumbai': {'lat': 19.0760, 'lon': 72.8777},
    'Patna': {'lat': 25.5941, 'lon': 85.1376},
    'Shillong': {'lat': 25.5788, 'lon': 91.8933},
    'Talcher': {'lat': 20.9509, 'lon': 85.2163},
    'Thiruvananthapuram': {'lat': 8.5241, 'lon': 76.9366},
    'Visakhapatnam': {'lat': 17.6868, 'lon': 83.2185}
}

weather_frames = []

# Fetch data using the complete list
for city in df['City'].unique():
    # Safety check in case a new city appears in future data
    if city not in city_coords:
        print(f"⚠️ Skipping {city}: Coordinates still missing.")
        continue
        
    lat = city_coords[city]['lat']
    lon = city_coords[city]['lon']
    
    # Get date range for this specific city
    city_data = df[df['City'] == city]
    start_date = city_data['Date'].min().strftime('%Y-%m-%d')
    end_date = city_data['Date'].max().strftime('%Y-%m-%d')
    
    print(f"Fetching data for {city}...")
    
    try:
        # Fetch Temperature, Humidity, and Wind Speed
        url = f"https://archive-api.open-meteo.com/v1/archive?latitude={lat}&longitude={lon}&start_date={start_date}&end_date={end_date}&daily=temperature_2m_mean,relative_humidity_2m_mean,wind_speed_10m_max&timezone=auto"
        
        response = requests.get(url)
        
        if response.status_code == 200:
            data = response.json()
            
            # Create a temporary dataframe for the weather data
            temp_df = pd.DataFrame({
                'Date': pd.to_datetime(data['daily']['time']),
                'City': city,
                'Temp_Mean': data['daily']['temperature_2m_mean'],
                'Humidity_Mean': data['daily']['relative_humidity_2m_mean'],
                'Wind_Speed_Max': data['daily']['wind_speed_10m_max']
            })
            weather_frames.append(temp_df)
        else:
            print(f"❌ Error fetching {city}: {response.status_code}")
            
    except Exception as e:
        print(f"❌ Exception for {city}: {e}")
    
    # Pause briefly to respect API rate limits
    time.sleep(1)

# Combine and Merge
if weather_frames:
    all_weather_df = pd.concat(weather_frames)
    
    # Merge with original data
    final_df = pd.merge(df, all_weather_df, on=['City', 'Date'], how='left')
    
    # Save to CSV
    final_df.to_csv('city_day_with_weather_complete.csv', index=False)
    print("\n✅ Success! All cities updated. Saved to 'city_day_with_weather_complete.csv'")
else:
    print("\n❌ No weather data fetched.")

In [ ]:
import numpy as np
df = pd.read_csv('city_day_with_weather_complete.csv')
df_numeric = df.select_dtypes(include=np.number)

df_numeric.head()
df.head()

In [ ]:
corr_matrix = df_numeric.corr()
print(corr_matrix)

corr_matrix = df_numeric.corr()
corr_matrix['AQI'].sort_values(ascending=False)
corr_target = corr_matrix['AQI'].abs()

selected_features = corr_target[corr_target > 0.3].index
selected_features

In [ ]:
X = df_numeric[selected_features]
y = df_numeric['AQI']
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,8))
sns.heatmap(corr_matrix, cmap='coolwarm', annot=True)
plt.show()

In [ ]:
# Convert Date column to datetime
df['Date'] = pd.to_datetime(df['Date'])

# Sort by Date (very important for time series)
df = df.sort_values('Date')
plt.figure()
plt.plot(df['Date'], df['AQI'])
plt.title("AQI Trend Over Time")
plt.xlabel("Date")
plt.ylabel("AQI")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# 1. Histogram - AQI
plt.figure()
df['AQI'].dropna().hist()
plt.title("Distribution of AQI")
plt.xlabel("AQI")
plt.ylabel("Frequency")
plt.show()


# 2. Scatter Plot - PM2.5 vs AQI
plt.figure()
plt.scatter(df['PM2.5'], df['AQI'])
plt.title("PM2.5 vs AQI")
plt.xlabel("PM2.5")
plt.ylabel("AQI")
plt.show()


# 3. Scatter Plot - PM10 vs AQI
plt.figure()
plt.scatter(df['PM10'], df['AQI'])
plt.title("PM10 vs AQI")
plt.xlabel("PM10")
plt.ylabel("AQI")
plt.show()


# 4. Boxplot - AQI (Outlier Detection)
plt.figure()
plt.boxplot(df['AQI'].dropna())
plt.title("Boxplot of AQI")
plt.ylabel("AQI")
plt.show()

plt.figure()
plt.scatter(df['Humidity_Mean'], df['AQI'])
plt.title("Scatter Plot: Humidity_Mean vs AQI")
plt.xlabel("Humidity_Mean")
plt.ylabel("AQI")
plt.show()

plt.figure()
df.boxplot(column='AQI', by='AQI_Bucket')
plt.title("AQI Distribution by AQI_Bucket")
plt.suptitle("")  # removes automatic subtitle
plt.xlabel("AQI_Bucket")
plt.ylabel("AQI")
plt.show()


# Bar Plot: Average AQI by City
# ----------------------------
plt.figure()
df.groupby('City')['AQI'].mean().plot(kind='bar')
plt.title("Average AQI by City")
plt.xlabel("City")
plt.ylabel("Average AQI")
plt.show()

In [ ]:
df = pd.read_csv('city_day_with_weather_complete.csv')

df = df.dropna(subset=['AQI', 'AQI_Bucket'])

columns_to_drop = [
    'AQI_Bucket',
    'NH3',
    'O3',
    'Benzene',
    'Toluene',
    'Xylene',
]

df = df.drop(columns=columns_to_drop)

In [ ]:
# 1. Remove all rows where City is 'Lucknow'
df = df[df['City'] != 'Lucknow']
df.isnull().sum()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Select only numeric columns with non-null values
numeric_cols = ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'CO', 'SO2']

ncols = 3
nrows = int(np.ceil(len(numeric_cols) / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 4))
axes = axes.flatten()

for idx, col in enumerate(numeric_cols):
    ax = axes[idx]
    data = df[col].dropna()  # only non-null values
    
    sns.histplot(data, ax=ax, kde=True, color='steelblue', bins=50, alpha=0.7)
    
    ax.set_title(f"{col}\n(n={len(data):,} non-null)", fontsize=11, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel("Frequency")
    ax.axvline(data.mean(),   color='red',    linestyle='--', linewidth=1.2, label=f'Mean: {data.mean():.1f}')
    ax.axvline(data.median(), color='orange', linestyle='--', linewidth=1.2, label=f'Median: {data.median():.1f}')
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

# Hide unused subplots
for j in range(len(numeric_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Distribution of All Numeric Columns (Non-Null Only)", 
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig("distributions.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: distributions.png")

In [ ]:
# Columns with nulls to fill
cols_to_fill = ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'CO', 'SO2']

# Fill nulls with median per city (better than global median)
for col in cols_to_fill:
    df[col] = df.groupby('City')[col].transform(
        lambda x: x.fillna(x.median())
    )

# Check if any nulls remain (edge case: entire city has no values)
remaining = df[cols_to_fill].isnull().sum()
print("Nulls remaining after city-level fill:")
print(remaining)

# If any still remain, fill with global median as fallback
for col in cols_to_fill:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)

print("\nFinal null check:")
print(df[cols_to_fill].isnull().sum())

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error

In [ ]:
df = df.sort_values(by=['City', 'Date'])

# All pollutant columns to create lag/roll for
poll_cols = ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'CO', 'SO2', 'AQI']

for col in poll_cols:
    grp = df.groupby('City')[col]
    
    # Lag features — what was the value 1, 3, 7 days ago?
    df[f'{col}_lag1'] = grp.shift(1)
    df[f'{col}_lag3'] = grp.shift(3)
    df[f'{col}_lag7'] = grp.shift(7)
    
    # Rolling mean — average trend over past 7 and 30 days
    df[f'{col}_roll7_mean']  = grp.shift(1).transform(
        lambda x: x.rolling(7,  min_periods=1).mean())
    df[f'{col}_roll30_mean'] = grp.shift(1).transform(
        lambda x: x.rolling(30, min_periods=1).mean())
    
    # Rolling std — how volatile has it been?
    df[f'{col}_roll7_std']   = grp.shift(1).transform(
        lambda x: x.rolling(7,  min_periods=1).std())

df.dropna(inplace=True)

In [ ]:
y = df['AQI']

# Features
X = df.drop(columns=['AQI'])

In [ ]:
X = X.copy()

X['Date'] = pd.to_datetime(X['Date'], errors='coerce')

X['year'] = X['Date'].dt.year
X['month'] = X['Date'].dt.month
X['day'] = X['Date'].dt.day
X['day_of_week'] = X['Date'].dt.dayofweek

X = X.drop(columns=['Date'])

df = df.sort_values(by=['City', 'Date'])

split_index = int(0.8 * len(X))

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print(f"Training set size: {len(X_train)}")
print(f"Testing set size: {len(X_test)}")

In [ ]:
categorical_features = ['City']
numeric_features = X.drop(columns=['City']).columns

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

ridge_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', Ridge(alpha=1.0))
])

rf_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42))
])

xgb_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(n_estimators=100, learning_rate=0.1))
])

In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score

models = {
    "Ridge Regression": ridge_model,
    "Random Forest": rf_model,
    "XGBoost": xgb_model
}

results = []

# Sklearn Models
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    results.append([name, mse, rmse, r2])


# Convert to DataFrame (BEST for report)
import pandas as pd

results_df = pd.DataFrame(results, columns=["Model", "MSE", "RMSE", "R2"])

print(results_df)

In [ ]:
print("Features used in X:")
print(X.columns.tolist())
print(f"\nTotal features: {X.shape[1]}")